# Tuần 04 — Tóm tắt dữ liệu (3/3): Tương quan và hồi quy tuyến tính

**UET.MAT1052 — Xác suất thống kê**
Biên soạn: ThS. Hoàng Hữu Bách — BM. Khoa học & Kỹ thuật tính toán - Khoa Công nghệ Thông tin, VNU-UET

---

Ba tuần qua ta tóm tắt một biến, hoặc một biến phân loại đi cùng một biến khác. Tuần này hỏi: **hai hay nhiều biến số liên hệ với nhau thế nào**, và làm sao nén mối liên hệ đó thành vài con số mà không nói sai?

Có hai công cụ chính. **Hệ số tương quan** đo hướng và độ mạnh của một xu hướng tuyến tính bằng một con số. **Mô hình tuyến tính** vẽ một đường thẳng (hoặc một mặt phẳng) qua dữ liệu, cho phép tính giá trị kỳ vọng của từng quan sát và xem nó lệch bao xa. Cả hai đều mạnh, và đều dễ bị đọc sai: tưởng $r = 0$ là không liên quan, tưởng hệ số hồi quy là tác động nhân quả, hay tin một đường thẳng bị kéo lệch bởi một điểm duy nhất.

Sau tuần này, bạn cần làm được:

1. Mô tả quan hệ giữa hai biến số bằng biểu đồ phân tán và hệ số tương quan; giải thích đúng ý nghĩa và giới hạn của hệ số tương quan Pearson.
2. Viết, đọc và diễn giải mô hình hồi quy tuyến tính đơn giản; phân biệt giá trị quan sát, giá trị đối chiếu và phần dư.
3. Hiểu tiêu chuẩn bình phương tối thiểu, suy ra hai điều kiện chuẩn và liên hệ chúng với $b_0$, $b_1$.
4. Chẩn đoán phi tuyến, điểm ảnh hưởng mạnh và cấu trúc nhóm bằng biểu đồ và biểu đồ phần dư.
5. Diễn giải hệ số trong hồi quy nhiều biến theo nghĩa "giữ các biến còn lại cố định"; xử lý biến phân loại bằng biến chỉ báo và mức tham chiếu.
6. Dùng Python để tính tương quan, ước lượng mô hình, đọc hệ số và phần dư; phân biệt mô tả mối liên hệ với kết luận nhân quả.

Mã bài tập và mức độ (1 = nhận biết, 2 = vận dụng, 3 = thử thách) giữ như trên lớp. Tự trả lời trước khi mở đáp án.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 30)
rng = np.random.default_rng(42)
DATA = "datasets/"

**Về dữ liệu nghèo đói và tốt nghiệp.** STAT 20 và OpenIntro dùng dữ liệu tỉ lệ tốt nghiệp trung học (`Graduates`) và tỉ lệ sống dưới ngưỡng nghèo (`Poverty`) của 50 bang Hoa Kỳ và Đặc khu Columbia. Bản dữ liệu gốc không tải được trong môi trường biên soạn, nên file `poverty_mo_phong.csv` là **dữ liệu mô phỏng** 51 hàng, được dựng sao cho khớp đúng các thống kê tóm tắt mà nguồn công bố ($\bar x = 86{,}01$, $\bar y = 11{,}35$, $s_x = 3{,}73$, $s_y = 3{,}10$, $r = -0{,}747$), và giữ đúng hai quan sát được dùng trong bài: California (81,1; 12,8) và Rhode Island (81; 10,3). Mọi đường hồi quy, hệ số tương quan và phần dư tính từ file này vì thế trùng với số liệu trong sách, sai khác chỉ do làm tròn. Tên các bang khác được thay bằng "Bang 01", "Bang 02"...

In [ ]:
poverty = pd.read_csv(DATA + "poverty_mo_phong.csv")
print(poverty.shape)
poverty.head()

---
## Phần 1. Từ biểu đồ phân tán đến hệ số tương quan

### 1.1 Mối liên hệ giữa hai biến số

Ở Tuần 02, hai biến phân loại có mối liên hệ nếu phân phối có điều kiện của biến này thay đổi theo mức của biến kia. Định nghĩa đó dùng được cho cả biến số: **hai biến có mối liên hệ nếu phân phối có điều kiện của một biến thay đổi khi đi qua các giá trị của biến kia.**

Trên biểu đồ phân tán, cách kiểm tra là quét từ trái sang phải. Hình dưới làm đúng việc đó: khoanh những bang nghèo ít (xanh) và những bang nghèo nhiều (đỏ), rồi vẽ phân phối tỉ lệ tốt nghiệp trong mỗi khung. Bên phải là cùng dữ liệu nhưng cột `Poverty` bị xáo trộn ngẫu nhiên, nên mọi liên hệ bị phá vỡ.

In [ ]:
xao_tron = poverty.assign(Poverty=rng.permutation(poverty["Poverty"].values))

fig, axes = plt.subplots(1, 4, figsize=(14, 4), gridspec_kw={"width_ratios": [3, 1, 3, 1]})
luoi_y = np.linspace(75, 95, 200)
for k, (du_lieu, tieu_de) in enumerate([(poverty, "Dữ liệu"), (xao_tron, "Poverty bị xáo trộn")]):
    ax, ax_mat_do = axes[2 * k], axes[2 * k + 1]
    ax.scatter(du_lieu["Poverty"], du_lieu["Graduates"], s=15, color="black")
    thap = du_lieu[du_lieu["Poverty"] < 9]["Graduates"]
    cao = du_lieu[du_lieu["Poverty"] > 14]["Graduates"]
    ax.axvspan(5, 9, color="tab:cyan", alpha=0.15)
    ax.axvspan(14, 19.5, color="tab:red", alpha=0.12)
    ax.set_xlabel("Tỉ lệ nghèo (%)")
    ax.set_ylabel("Tỉ lệ tốt nghiệp (%)")
    ax.set_title(tieu_de)
    ax_mat_do.plot(stats.gaussian_kde(thap)(luoi_y), luoi_y, color="tab:cyan", label="nghèo < 9%")
    ax_mat_do.plot(stats.gaussian_kde(cao)(luoi_y), luoi_y, color="tab:red", label="nghèo > 14%")
    ax_mat_do.set_xticks([])
    ax_mat_do.set_ylim(ax.get_ylim())
axes[1].legend(fontsize=7, loc="lower right")
plt.suptitle("Hình 1.1. Phân phối có điều kiện dịch chuyển (trái) hay không (phải)")
plt.tight_layout()
plt.show()

Ở dữ liệu thật (trái), khi đi từ khung xanh sang khung đỏ, phân phối tỉ lệ tốt nghiệp trượt xuống: các bang nghèo ít thường có tỉ lệ tốt nghiệp 85–90%, các bang nghèo nhiều chỉ ở mức thấp 80%. Ở dữ liệu xáo trộn (phải), hai phân phối gần như chồng lên nhau: không có mối liên hệ.

Khi đọc một biểu đồ phân tán, hỏi bốn câu:

1. **Hướng:** dương hay âm?
2. **Dạng:** tuyến tính hay phi tuyến?
3. **Độ mạnh:** các điểm bám sát một cấu trúc đến mức nào?
4. **Quan sát ngoại lai:** có điểm nào khác hẳn đám mây dữ liệu không?

Một con số tóm tắt chỉ có nghĩa sau khi đã nhìn cấu trúc dữ liệu.

### 1.2 Tỉ lệ nghèo và tỉ lệ tốt nghiệp

Từ đây trở đi, theo cách trình bày của OpenIntro, coi `Poverty` là **biến phụ thuộc** (response, trục y) và `Graduates` là **biến giải thích** (trục x).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(poverty["Graduates"], poverty["Poverty"], s=25, color="#4C9FC7", alpha=0.8)
ax.set_xlabel("Tỉ lệ tốt nghiệp trung học (%)")
ax.set_ylabel("Tỉ lệ nghèo (%)")
ax.set_title("Hình 1.2. 51 bang: tốt nghiệp và nghèo")
plt.show()

Mối liên hệ có hướng âm, khá tuyến tính, độ mạnh vừa tới khá.

> **W04-Q01 · Mức 1 — Ước lượng hệ số tương quan.** Từ Hình 1.2, giá trị nào hợp lý nhất cho hệ số tương quan giữa `Poverty` và `Graduates`? (a) 0,60 (b) −0,75 (c) −0,10 (d) 0,02 (e) −1,50. Giải thích bằng hướng, độ mạnh và miền giá trị hợp lệ.

<details><summary>Đáp án</summary>

(b) −0,75. Dấu âm khớp với xu hướng đi xuống; độ lớn khoảng 0,75 khớp với một liên hệ tuyến tính khá mạnh nhưng không hoàn hảo; và hệ số tương quan luôn nằm trong $[-1, 1]$ nên −1,50 là không thể. STAT 20 tính được $r \approx -0{,}747$.
</details>

### 1.3 Hệ số tương quan Pearson

Muốn một con số đo "các điểm cùng đi theo một đường thẳng đến mức nào", ta cần nó: không phụ thuộc đơn vị đo, dương khi hai biến cùng tăng, âm khi biến này tăng biến kia giảm, và lớn khi các điểm bám sát một đường thẳng. Công thức sau thỏa tất cả:

$$
r = \frac{1}{n-1}\sum_{i=1}^{n}\left(\frac{x_i - \bar{x}}{s_x}\right)\left(\frac{y_i - \bar{y}}{s_y}\right).
$$

Đọc công thức từng lớp:

- Mỗi biến được **chuẩn hóa**: $z_{x,i} = (x_i - \bar x)/s_x$ cho biết quan sát $i$ cách trung bình bao nhiêu độ lệch chuẩn. Việc này xóa đơn vị đo.
- Nếu một bang vừa tốt nghiệp cao hơn trung bình vừa nghèo cao hơn trung bình, hai độ lệch **cùng dấu**, tích **dương**. Nếu trái dấu, tích **âm**.
- Cộng các tích lại (và chia $n-1$): nếu phần lớn điểm nằm ở hai góc "cùng dấu", $r$ dương; ở hai góc "trái dấu", $r$ âm.

Hình dưới tô màu từng điểm theo dấu của tích $z_x z_y$.

In [ ]:
x = poverty["Graduates"]
y = poverty["Poverty"]
z_x = (x - x.mean()) / x.std()
z_y = (y - y.mean()) / y.std()
tich = z_x * z_y

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(z_x, z_y, c=np.where(tich > 0, "tab:blue", "tab:red"), s=30)
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.text(1.5, 2.2, "tích > 0", color="tab:blue")
ax.text(-2.4, 2.2, "tích < 0", color="tab:red")
ax.set_xlabel("Tỉ lệ tốt nghiệp đã chuẩn hóa")
ax.set_ylabel("Tỉ lệ nghèo đã chuẩn hóa")
ax.set_title("Hình 1.3. Mỗi điểm góp một tích z_x · z_y vào r")
plt.show()

r_tay = tich.sum() / (len(x) - 1)
print("r tính theo công thức:", round(r_tay, 3))
print("r theo pandas         :", round(poverty["Poverty"].corr(poverty["Graduates"]), 3))

Phần lớn điểm nằm ở góc trên trái và dưới phải (màu đỏ), nên tổng các tích âm và $r \approx -0{,}747$.

**Những gì $r$ nói được, và không nói được:**

- $-1 \le r \le 1$; $r$ không có đơn vị.
- $r > 0$: liên hệ tuyến tính dương; $r < 0$: liên hệ tuyến tính âm.
- $|r|$ càng gần 1, các điểm càng gần một đường thẳng.
- $r = 0$ chỉ có nghĩa là **không có liên hệ tuyến tính** theo thước đo này. Không được rút gọn thành "hai biến không liên quan".

> **W04-EX01 · Mức 2 — Đổi đơn vị có đổi tương quan?** Thay $x$ bởi $x' = a + bx$. Dự đoán $r(x', y)$ so với $r(x, y)$ khi: (1) chỉ cộng hằng số ($b = 1$); (2) nhân với hệ số dương ($b > 0$); (3) đảo chiều thang đo ($b < 0$). Lập luận từ phép chuẩn hóa, không tính lại từ đầu.

In [ ]:
for a, b in [(10, 1), (0, 0.01), (100, -1)]:
    x_moi = a + b * poverty["Graduates"]
    print(f"a = {a:>4}, b = {b:>5}:  r = {poverty['Poverty'].corr(x_moi):.3f}")

<details><summary>Đáp án</summary>

1. Cộng hằng số không đổi tương quan: độ lệch $x_i - \bar x$ giữ nguyên.
2. Nhân số dương không đổi tương quan: $x_i - \bar x$ và $s_x$ cùng nhân $b$, tỉ số giữ nguyên.
3. Nhân số âm đổi dấu tương quan: mọi độ lệch chuẩn hóa của $x$ đổi dấu, còn $s_x$ vẫn dương.

Ví dụ ở dòng thứ hai: đổi `Graduates` từ phần trăm sang tỉ lệ 0–1 ($b = 0{,}01$) không làm $r$ thay đổi.
</details>

> **W04-CH01 · Mức 3 — Tự xây phản ví dụ cho "$r = 0$".** Chứng minh hoặc bác bỏ: "Nếu $r = 0$, biết $x$ không cung cấp thông tin có cấu trúc nào về $y$." (1) Tự xây một bộ dữ liệu nhỏ có quan hệ xác định mạnh giữa $x$ và $y$ nhưng tương quan bằng 0 hoặc rất gần 0. (2) Kiểm tra cấu hình $x = (-2, -1, 0, 1, 2)$, $y = (4, 1, 0, 1, 4)$.

In [ ]:
x_ch = np.array([-2, -1, 0, 1, 2])
y_ch = np.array([4, 1, 0, 1, 4])
print("r =", np.corrcoef(x_ch, y_ch)[0, 1])

fig, ax = plt.subplots(figsize=(4.5, 3.5))
luoi = np.linspace(-2.3, 2.3, 100)
ax.plot(luoi, luoi ** 2, color="lightgray")
ax.scatter(x_ch, y_ch, color="black", zorder=3)
ax.set(xlabel="x", ylabel="y", title="y = x²: quan hệ hoàn hảo, r = 0")
plt.show()

<details><summary>Lời giải</summary>

Khẳng định sai. Ở đây $y = x^2$: biết $x$ là biết chính xác $y$, nhưng quan hệ phi tuyến và đối xứng quanh $x = 0$. Vì $\bar x = 0$ và $\sum x_i y_i = \sum x_i^3 = 0$, tổng các tích độ lệch bằng 0, nên $r = 0$.

Kết luận đúng: $r = 0$ nghĩa là không thấy liên hệ tuyến tính; nó không loại trừ một cấu trúc phi tuyến mạnh. Đó là lý do phải nhìn biểu đồ trước.
</details>

> **W04-EX02 · Mức 2 — Một điểm có thể đổi cả $r$ lẫn đường hồi quy.** *Dữ liệu minh họa.* Năm điểm đầu có $x = (1, 2, 3, 4, 5)$, $y = (1{,}2;\ 1{,}8;\ 3{,}1;\ 3{,}9;\ 5{,}1)$, với $r \approx 0{,}995$ và $\hat y \approx 0{,}05 + 0{,}99x$. Thêm điểm $(10, 0)$. Không dùng phần mềm, dự đoán: (1) dấu của tương quan và hệ số góc có thể đổi không? (2) điểm mới có đòn bẩy cao không? (3) một đường hồi quy duy nhất sau khi thêm điểm có còn mô tả tốt 5 điểm ban đầu không?

In [ ]:
x5 = np.array([1, 2, 3, 4, 5.0])
y5 = np.array([1.2, 1.8, 3.1, 3.9, 5.1])
x6 = np.append(x5, 10)
y6 = np.append(y5, 0)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x5, y5, color="black", label="5 điểm ban đầu")
ax.scatter([10], [0], color="red", s=70, label="điểm thêm (10, 0)")
luoi = np.linspace(0, 11, 50)
for xs, ys, nhan, kieu in [(x5, y5, "hồi quy, 5 điểm", "-"), (x6, y6, "hồi quy, 6 điểm", "--")]:
    b1, b0 = np.polyfit(xs, ys, 1)
    ax.plot(luoi, b0 + b1 * luoi, kieu, label=f"{nhan}: ŷ = {b0:.2f} + ({b1:.2f})x, r = {np.corrcoef(xs, ys)[0, 1]:.3f}")
ax.set(xlabel="x", ylabel="y", title="Hình 1.4. Một điểm có đòn bẩy cao")
ax.legend(fontsize=8)
plt.show()

<details><summary>Đáp án</summary>

Sau khi thêm $(10, 0)$: $r \approx -0{,}259$ và $\hat y \approx 3{,}149 - 0{,}152x$. Cả dấu của $r$ lẫn hệ số góc đều đảo từ dương sang âm. Điểm mới nằm rất xa trung tâm theo trục $x$ nên có **đòn bẩy cao** và ảnh hưởng mạnh tới đường hồi quy. Đường mới mô tả rất kém xu hướng gần tuyến tính dương của 5 điểm đầu.

Luôn xem biểu đồ phân tán và so sánh mô hình có và không có điểm ảnh hưởng mạnh trước khi tin một tóm tắt tuyến tính.
</details>

---
## Phần 2. Mô hình hồi quy tuyến tính đơn giản

### 2.1 Một đường thẳng để tóm tắt dữ liệu

Cách thứ hai để tóm tắt một liên hệ tuyến tính đơn giản là... vẽ một đường thẳng. Đường thẳng vừa là tóm tắt bằng hình, vừa là tóm tắt bằng số, vì mỗi đường thẳng được xác định bởi hai con số. **Mô hình tuyến tính đơn giản** có dạng

$$
\hat{y} = b_0 + b_1 x,
$$

trong đó $\hat y$ là **giá trị đối chiếu** (fitted value) của biến phụ thuộc, $b_0$ là **hệ số chặn**, $b_1$ là **hệ số góc**, $x$ là biến giải thích. Hai con số $b_0$, $b_1$ nén một xu hướng hai biến thành một đường thẳng.

Thử vẽ bằng mắt trước. Ô dưới đặt một đường "vẽ tay" qua đám mây điểm. Bạn có thể sửa `b0_tay`, `b1_tay` để đường đi qua giữa đám mây tốt hơn.

In [ ]:
# Thử thay đổi: một đường thẳng vẽ bằng mắt
b0_tay = 60
b1_tay = -0.57

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(poverty["Graduates"], poverty["Poverty"], s=20, color="black")
luoi = np.linspace(77, 93, 50)
ax.plot(luoi, b0_tay + b1_tay * luoi, color="purple")
ax.set(xlabel="Tỉ lệ tốt nghiệp (%)", ylabel="Tỉ lệ nghèo (%)",
       title=f"Hình 2.1. Đường vẽ bằng mắt: ŷ = {b0_tay} + ({b1_tay})x")
plt.show()

Vẽ bằng mắt thì mỗi người một đường. Phần 3 sẽ đưa ra một tiêu chuẩn chính xác. Tạm thời, OpenIntro cho mô hình xấp xỉ

$$
\widehat{Poverty} = 64{,}68 - 0{,}62 \times Graduates.
$$

**Diễn giải hệ số góc:** so sánh các bang có tỉ lệ tốt nghiệp chênh nhau 1 điểm phần trăm, bang cao hơn có tỉ lệ nghèo dự báo bởi mô hình thấp hơn khoảng 0,62 điểm phần trăm. Đây là mô tả mối liên hệ trong dữ liệu, chưa phải kết luận nhân quả.

> **W04-Q02 · Mức 1 — Tính một giá trị đối chiếu.** Theo mô hình trên, một bang có `Graduates` = 85,1 được dự báo có tỉ lệ nghèo khoảng bao nhiêu? Đó là giá trị quan sát hay giá trị đối chiếu?

<details><summary>Đáp án</summary>

$\widehat{Poverty} = 64{,}68 - 0{,}62 \times 85{,}1 \approx 11{,}92$. Đây là **giá trị đối chiếu** từ mô hình, không phải tỉ lệ nghèo thực tế của bang đó.
</details>

### 2.2 Hệ số góc phụ thuộc đơn vị, tương quan thì không

Đổi `Graduates` từ "điểm phần trăm" sang "tỉ lệ từ 0 đến 1": các giá trị $x$ nhỏ đi 100 lần, nên hệ số góc phải **lớn lên** 100 lần để biểu diễn cùng một đường thẳng. Còn $r$ không đổi (W04-EX01). Vì vậy **không được dùng độ lớn của hệ số góc làm thước đo độ mạnh của mối liên hệ** nếu bỏ qua đơn vị.

In [ ]:
for ten, x_dv in [("phần trăm", poverty["Graduates"]), ("tỉ lệ 0–1", poverty["Graduates"] / 100)]:
    b1, b0 = np.polyfit(x_dv, poverty["Poverty"], 1)
    print(f"Graduates theo {ten:>9}: b1 = {b1:8.2f},  r = {poverty['Poverty'].corr(x_dv):.3f}")

### 2.3 Hệ số chặn có luôn đáng diễn giải?

Trong mô hình trên, $b_0 = 64{,}68$ là giá trị đối chiếu khi `Graduates` = 0. Nhưng không bang nào có tỉ lệ tốt nghiệp gần 0%. Hệ số chặn cần thiết về mặt đại số (để xác định đường thẳng), nhưng diễn giải thực tế của nó ở đây không đáng tin và không hữu ích.

> **W04-EX03 · Mức 2 — Có nên diễn giải hệ số chặn?** Một sinh viên nói: "Nếu một bang có 0% dân số tốt nghiệp trung học, mô hình chứng minh tỉ lệ nghèo sẽ là 64,68%." Chỉ ra hai lỗi.

<details><summary>Đáp án</summary>

1. `Graduates` = 0 nằm rất xa phạm vi dữ liệu (khoảng 78–92%): đây là **ngoại suy**.
2. Mô hình mô tả một mối liên hệ; nó không "chứng minh" một giá trị thực tế hay một tác động nhân quả.

Diễn giải an toàn hơn: 64,68 là chỗ đường hồi quy cắt trục tung, nhưng không có ý nghĩa thực tiễn đáng tin với dữ liệu này.
</details>

---
## Phần 3. Phần dư và bình phương tối thiểu

### 3.1 Phần dư: phần dữ liệu còn lại sau mô hình

Mỗi quan sát được tách thành hai phần:

$$
\text{dữ liệu} = \text{giá trị đối chiếu} + \text{phần dư}, \qquad e_i = y_i - \hat{y}_i .
$$

- $e_i > 0$: điểm nằm **trên** đường hồi quy; quan sát cao hơn mô hình kỳ vọng.
- $e_i < 0$: điểm nằm **dưới** đường hồi quy.
- $|e_i|$ càng lớn: mô hình khớp quan sát đó càng kém.

Về hình học, phần dư là **khoảng cách thẳng đứng** từ điểm tới đường hồi quy.

In [ ]:
b1_ls, b0_ls = np.polyfit(poverty["Graduates"], poverty["Poverty"], 1)
khop = b0_ls + b1_ls * poverty["Graduates"]

fig, ax = plt.subplots(figsize=(7, 5))
ax.vlines(poverty["Graduates"], khop, poverty["Poverty"], color="tab:green", alpha=0.6, linewidth=1)
ax.scatter(poverty["Graduates"], poverty["Poverty"], s=25, color="#4C9FC7", zorder=3)
luoi = np.linspace(77, 93, 50)
ax.plot(luoi, b0_ls + b1_ls * luoi, color="tab:red", linewidth=2)
ax.set(xlabel="Tỉ lệ tốt nghiệp (%)", ylabel="Tỉ lệ nghèo (%)",
       title="Hình 3.1. Phần dư là các đoạn thẳng đứng từ điểm tới đường hồi quy")
plt.show()

> **W04-WP01 · Mức 2 — Tính và diễn giải phần dư.** Rhode Island có `Graduates` = 81, `Poverty` = 10,3. Với mô hình $\widehat{Poverty} = 64{,}68 - 0{,}62\, Graduates$: (1) tính $\hat y$; (2) tính phần dư $e$; (3) diễn giải dấu và độ lớn của phần dư trong ngữ cảnh.

<details><summary>Lời giải</summary>

1. $\hat y = 64{,}68 - 0{,}62 \times 81 = 14{,}46$.
2. $e = y - \hat y = 10{,}3 - 14{,}46 = -4{,}16$.
3. Tỉ lệ nghèo quan sát được của Rhode Island thấp hơn giá trị đối chiếu của mô hình khoảng 4,16 **điểm phần trăm**. Dấu âm không có nghĩa là "sai số âm 4,16%"; đơn vị là điểm phần trăm của biến phụ thuộc.
</details>

STAT 20 tính phần dư cho California theo chiều ngược lại, dùng `Poverty` làm biến giải thích và `Graduates` làm biến phụ thuộc. Đường hồi quy khi đó là $\widehat{Graduates} = 96{,}20 - 0{,}898 \times Poverty$. California có `Poverty` = 12,8 nên $\hat y = 84{,}71$, trong khi tỉ lệ tốt nghiệp thật là 81,1: phần dư khoảng −3,6. Trong số các bang có tỉ lệ nghèo quanh 12,8%, ta kỳ vọng tỉ lệ tốt nghiệp khoảng 84,7%, nhưng California thấp hơn 3,6 điểm.

In [ ]:
b1_nguoc, b0_nguoc = np.polyfit(poverty["Poverty"], poverty["Graduates"], 1)
cali = poverty[poverty["State"] == "California"].iloc[0]
y_hat_cali = b0_nguoc + b1_nguoc * cali["Poverty"]
print(f"Graduates ~ Poverty: ŷ = {b0_nguoc:.2f} + ({b1_nguoc:.3f})x")
print(f"California: ŷ = {y_hat_cali:.2f}, y = {cali['Graduates']}, phần dư = {cali['Graduates'] - y_hat_cali:.2f}")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(poverty["Poverty"], poverty["Graduates"], s=20, color="black")
luoi = np.linspace(5, 19.5, 50)
ax.plot(luoi, b0_nguoc + b1_nguoc * luoi, color="blue")
ax.hlines(y_hat_cali, 5, cali["Poverty"], linestyle="--", color="gray")
ax.vlines(cali["Poverty"], cali["Graduates"], y_hat_cali, linestyle="--", color="red")
ax.annotate("California (12,8; 81,1)", xy=(cali["Poverty"], cali["Graduates"]), xytext=(13.5, 79))
ax.set(xlabel="Tỉ lệ nghèo (%)", ylabel="Tỉ lệ tốt nghiệp (%)", title="Hình 3.2. Phần dư của California")
plt.show()

Để ý hai đường hồi quy "y theo x" và "x theo y" **không phải** là cùng một đường vẽ ngược lại: hệ số góc −0,62 và −0,898 không nghịch đảo nhau. Mỗi đường giảm phần dư theo một chiều khác nhau. Phải chọn biến phụ thuộc theo câu hỏi.

### 3.2 Từ "một hằng số tốt nhất" đến hồi quy

Tuần 02 đã giải một bài toán đơn giản hơn: chọn một hằng số $a$ đại diện cho mọi $y_i$ sao cho tổng bình phương sai số nhỏ nhất,

$$
S(a) = \sum_{i=1}^{n}(y_i - a)^2, \qquad \frac{dS}{da} = -2\sum_{i=1}^{n}(y_i - a) = 0 \;\Longrightarrow\; a = \bar y .
$$

Trung bình mẫu là **mô hình hằng số** tối ưu theo tiêu chuẩn tổng bình phương. Hồi quy tuyến tính mở rộng đúng ý đó: thay vì chọn một số, chọn hai số $b_0$, $b_1$ để giảm

$$
SSE = \sum_{i=1}^{n}\bigl[y_i - (b_0 + b_1 x_i)\bigr]^2 .
$$

Đó là **phương pháp bình phương tối thiểu**. Ô dưới tính $SSE$ trên một lưới rất nhiều cặp $(b_0, b_1)$ và vẽ bản đồ đường mức. Điểm thấp nhất chính là đường hồi quy.

In [ ]:
def sse(b0, b1):
    return ((poverty["Poverty"] - (b0 + b1 * poverty["Graduates"])) ** 2).sum()

luoi_b1 = np.linspace(-1.0, -0.25, 120)
luoi_b0 = np.linspace(30, 100, 120)
bang_sse = np.array([[sse(b0, b1) for b1 in luoi_b1] for b0 in luoi_b0])

fig, ax = plt.subplots(figsize=(6.5, 5))
muc = ax.contour(luoi_b1, luoi_b0, np.log(bang_sse), levels=25, cmap="viridis")
ax.scatter([b1_ls], [b0_ls], color="red", zorder=3, label=f"cực tiểu: b0 = {b0_ls:.2f}, b1 = {b1_ls:.3f}")
ax.scatter([b1_tay], [b0_tay], color="purple", marker="x", s=70, label="đường vẽ bằng mắt")
ax.set(xlabel="hệ số góc b1", ylabel="hệ số chặn b0", title="Hình 3.3. Đường mức của log(SSE) theo (b0, b1)")
ax.legend(fontsize=8)
plt.show()
print("SSE tại đường vẽ bằng mắt:", round(sse(b0_tay, b1_tay), 1), "  SSE nhỏ nhất:", round(sse(b0_ls, b1_ls), 1))

Các đường mức là những hình elip dẹt, kéo dài theo một hướng chéo: tăng $b_1$ một chút có thể được bù bằng giảm $b_0$, vì dữ liệu nằm xa gốc tọa độ. Cực tiểu duy nhất nằm ở tâm các elip.

**Vì sao bình phương phần dư?** Có hai tiêu chuẩn tự nhiên, $\sum |e_i|$ và $\sum e_i^2$. Bình phương tối thiểu phạt mạnh hơn các phần dư lớn, có cấu trúc đại số thuận tiện (lấy đạo hàm được, có nghiệm đóng), được hỗ trợ rộng rãi trong phần mềm, và là nền tảng cho nhiều kết quả hồi quy sau này. Tuần này dùng nó như một tiêu chuẩn khớp mô tả.

### 3.3 Hai điều kiện chuẩn

Đặt $S(a, b) = \sum_{i}(y_i - a - b x_i)^2$ và $e_i = y_i - a - b x_i$. Tại cực tiểu, hai đạo hàm riêng bằng 0:

$$
\frac{\partial S}{\partial a} = -2\sum_i e_i = 0, \qquad \frac{\partial S}{\partial b} = -2\sum_i x_i e_i = 0 .
$$

Vậy đường bình phương tối thiểu thỏa

$$
\boxed{\sum_i e_i = 0}, \qquad \boxed{\sum_i x_i e_i = 0}.
$$

Diễn giải: phần dư cân bằng quanh 0, và không còn xu hướng tuyến tính nào theo $x$ trong phần dư.

**Từ điều kiện chuẩn đến $b_0$, $b_1$.** Từ $\sum e_i = 0$ được $\bar y = b_0 + b_1 \bar x$, tức $b_0 = \bar y - b_1 \bar x$: đường hồi quy luôn đi qua điểm $(\bar x, \bar y)$. Thế vào điều kiện thứ hai và gom theo độ lệch quanh trung bình:

$$
b_1 = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sum_i (x_i - \bar x)^2} = r\,\frac{s_y}{s_x}.
$$

Vì $s_x, s_y > 0$, **dấu của $b_1$ luôn cùng dấu với $r$**.

In [ ]:
phan_du = poverty["Poverty"] - khop
print("Tổng phần dư            :", round(phan_du.sum(), 10))
print("Tổng x_i · e_i          :", round((poverty["Graduates"] * phan_du).sum(), 8))
print("Đường hồi quy tại x̄        :", round(b0_ls + b1_ls * poverty["Graduates"].mean(), 4), " = ȳ =", round(poverty["Poverty"].mean(), 4))

> **W04-WP02 · Mức 2 — Suy ra đường hồi quy từ thống kê tóm tắt.** Với dữ liệu nghèo và tốt nghiệp, OpenIntro cho $\bar x = 86{,}01$, $\bar y = 11{,}35$, $s_x = 3{,}73$, $s_y = 3{,}10$, $r = -0{,}75$. (1) Tính $b_1$. (2) Tính $b_0$. (3) Kiểm tra dấu của $b_1$ có khớp với $r$ không.

<details><summary>Lời giải</summary>

1. $b_1 = r\,\dfrac{s_y}{s_x} = -0{,}75 \times \dfrac{3{,}10}{3{,}73} \approx -0{,}62$.
2. $b_0 = \bar y - b_1 \bar x = 11{,}35 - (-0{,}62)(86{,}01) \approx 64{,}68$.
3. Vậy $\widehat{Poverty} = 64{,}68 - 0{,}62\, Graduates$. Dấu âm của $b_1$ khớp với $r < 0$.

Máy tính trên dữ liệu (Hình 3.3) cho $b_0 \approx 64{,}80$, $b_1 \approx -0{,}621$. Sai khác nhỏ ở $b_0$ đến từ việc làm tròn $r$ thành −0,75 và $b_1$ thành −0,62 trước khi nhân với $\bar x \approx 86$.
</details>

### 3.4 Ngoại suy: đường thẳng không có "giấy phép" vô hạn

Một mô hình tuyến tính chỉ được ước lượng trên phạm vi dữ liệu đã quan sát. Dùng nó rất xa ngoài phạm vi đó đòi hỏi một giả định mạnh: cấu trúc gần tuyến tính vẫn tiếp tục đúng ở vùng chưa có dữ liệu. Giả định đó thường không kiểm tra được bằng chính dữ liệu hiện có.

> **W04-EX04 · Mức 2 — Ngoại suy có đáng tin?** Mô hình nghèo–tốt nghiệp được xây từ các bang có tỉ lệ tốt nghiệp khoảng 78–92%. Một người dùng mô hình để "ước lượng" `Poverty` khi `Graduates` = 20. Đánh giá phép tính này theo hai lớp: (1) đại số: có tính được không? (2) thống kê: có đáng tin không?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(poverty["Graduates"], poverty["Poverty"], s=15, color="black")
luoi = np.linspace(15, 95, 100)
ax.plot(luoi, b0_ls + b1_ls * luoi, color="tab:red")
ax.axvspan(poverty["Graduates"].min(), poverty["Graduates"].max(), color="tab:blue", alpha=0.1, label="phạm vi dữ liệu")
ax.scatter([20], [b0_ls + b1_ls * 20], color="tab:red", marker="x", s=80, label=f"x = 20: ŷ ≈ {b0_ls + b1_ls * 20:.1f}%")
ax.set(xlabel="Tỉ lệ tốt nghiệp (%)", ylabel="Tỉ lệ nghèo (%)", title="Hình 3.4. Ngoại suy xa khỏi đám mây dữ liệu")
ax.legend(fontsize=8)
plt.show()

<details><summary>Đáp án</summary>

1. Đại số: thay $x = 20$ vào phương trình thì luôn ra một con số (khoảng 52%).
2. Thống kê: kết quả rất đáng nghi, vì $x = 20$ nằm cực xa vùng dữ liệu. Không có bang nào như vậy để kiểm tra xem quan hệ có còn tuyến tính không.

Một con số do phương trình trả về không đồng nghĩa với một ước lượng có cơ sở dữ liệu.
</details>

### 3.5 Biểu đồ phần dư: mô hình còn bỏ sót gì?

Nếu phần dư còn một hình dạng có hệ thống theo $x$, đường thẳng chưa tóm tắt hết cấu trúc. **Biểu đồ phần dư** vẽ $e_i$ theo $x_i$ (hoặc theo $\hat y_i$). Tuần này chỉ dùng nó như công cụ chẩn đoán mô tả; suy luận hồi quy sẽ học sau.

Ô dưới **mô phỏng** ba bộ dữ liệu như hình trong OpenIntro Statistics 4e: hàng trên là biểu đồ phân tán kèm đường hồi quy, hàng dưới là phần dư tương ứng.

In [ ]:
n = 40
x1 = rng.uniform(0, 10, n); y1 = 20 - 1.8 * x1 + rng.normal(0, 1.2, n)
x2 = rng.uniform(0, 10, n); y2 = 22 - 3.2 * x2 + 0.16 * x2 ** 2 + rng.normal(0, 0.5, n)
x3 = rng.uniform(1, 10, n); y3 = 5 + 0.6 * x3 + rng.normal(0, 0.8 * x3, n)
bo_du_lieu = [(x1, y1, "tuyến tính"), (x2, y2, "cong"), (x3, y3, "phễu")]

fig, axes = plt.subplots(2, 3, figsize=(12, 5.5), gridspec_kw={"height_ratios": [2, 1]})
for j, (xs, ys, ten) in enumerate(bo_du_lieu):
    b1, b0 = np.polyfit(xs, ys, 1)
    e = ys - (b0 + b1 * xs)
    luoi = np.linspace(xs.min(), xs.max(), 50)
    axes[0, j].scatter(xs, ys, s=12, color="#4C9FC7")
    axes[0, j].plot(luoi, b0 + b1 * luoi, color="black")
    axes[0, j].set_title(ten)
    axes[1, j].scatter(xs, e, s=12, color="#4C9FC7")
    axes[1, j].axhline(0, linestyle="--", color="black")
    axes[1, j].set_xlabel("x")
axes[0, 0].set_ylabel("y")
axes[1, 0].set_ylabel("phần dư")
plt.suptitle("Hình 3.5. Biểu đồ phân tán (trên) và biểu đồ phần dư (dưới), dữ liệu mô phỏng")
plt.tight_layout()
plt.show()

> **W04-CH04 · Mức 3 — Chẩn đoán từ biểu đồ phần dư.** Một mô hình tuyến tính phù hợp nên để phần dư rải quanh 0 mà không còn cấu trúc có hệ thống. Chẩn đoán: (1) đường cong rõ rệt: giả định nào về dạng quan hệ đang thất bại? (2) dạng "phễu": mô hình bỏ sót điều gì về độ biến thiên? (3) đám mây ngẫu nhiên quanh 0 có đủ để kết luận nhân quả không?

<details><summary>Lời giải</summary>

1. Đường cong có hệ thống (Hình 3.5, giữa) cho thấy dạng tuyến tính chưa tóm tắt hết quan hệ; cần xem xét cấu trúc phi tuyến. Để ý ở hàng trên, đường thẳng trông khá ổn: biểu đồ phần dư phóng to cái mà mắt dễ bỏ qua.
2. Dạng phễu (phải) cho thấy mức dao động của $y$ quanh đường hồi quy tăng theo $x$; một mức phân tán cố định là mô tả kém.
3. Không. Biểu đồ phần dư chỉ chẩn đoán chất lượng của tóm tắt tuyến tính, không cung cấp thiết kế nhân quả.

Biểu đồ phần dư đẹp không chứng minh mô hình "đúng"; nó chỉ cho biết chưa thấy vài kiểu thất bại rõ ràng.
</details>

---
## Phần 4. Hồi quy tuyến tính đa biến

### 4.1 Từ một biến giải thích đến nhiều biến

Hầu hết hiện tượng thật liên quan tới nhiều biến cùng lúc. **Hồi quy tuyến tính đa biến** mở rộng mô hình thành

$$
\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \dots + b_p x_p .
$$

Mỗi hệ số trả lời một câu hỏi **có điều kiện** trên các biến đang có trong mô hình: khi $x_j$ tăng 1 đơn vị, giá trị đối chiếu của $y$ thay đổi bao nhiêu, **giữ các biến giải thích khác cố định**?

Các hệ số vẫn được chọn để tối thiểu hóa $SSE = \sum_i (y_i - \hat y_i)^2$ với $\hat y_i = b_0 + b_1 x_{i1} + \dots + b_p x_{ip}$. Ý tưởng không đổi; chỉ là không gian tìm kiếm có nhiều chiều hơn.

### 4.2 Một biến giải thích phân loại: biến chỉ báo

Phép nhân chỉ định nghĩa cho số, vậy đưa một biến phân loại vào mô hình tuyến tính thế nào? Mẹo là **biến chỉ báo** (indicator variable, còn gọi là biến giả, dummy variable): một biến nhận 1 nếu quan sát thuộc một mức cụ thể, 0 nếu không. Một biến phân loại có $k$ mức cần $k - 1$ biến chỉ báo; mức không có biến chỉ báo gọi là **mức tham chiếu** (reference level).

Ví dụ của OpenIntro dùng 15 cuốn sách (dữ liệu `allbacks`): `weight` (khối lượng, g), `volume` (thể tích, cm³), `cover` (bìa cứng `hb` hoặc bìa mềm `pb`). Đặt

$$
D = \begin{cases} 0, & \text{bìa cứng (mức tham chiếu)} \\ 1, & \text{bìa mềm.} \end{cases}
$$

In [ ]:
books = pd.read_csv(DATA + "allbacks.csv")
books["D"] = (books["cover"] == "pb").astype(int)

X = np.column_stack([np.ones(len(books)), books["volume"], books["D"]])
b0_s, b_vol, b_D = np.linalg.lstsq(X, books["weight"], rcond=None)[0]
print(f"weight^ = {b0_s:.2f} + {b_vol:.2f}·volume + ({b_D:.2f})·D")

Mô hình

$$
\widehat{weight} = 197{,}96 + 0{,}72\, volume - 184{,}05\, D
$$

có một biến phụ thuộc số (`weight`), một biến giải thích số (`volume`) và một biến giải thích phân loại hai mức (`cover`). Thay hai giá trị của $D$:

$$
D = 0 \text{ (bìa cứng)}: \ \widehat{weight} = 197{,}96 + 0{,}72\, volume, \qquad
D = 1 \text{ (bìa mềm)}: \ \widehat{weight} = 13{,}91 + 0{,}72\, volume .
$$

Hai đường **song song**: cùng hệ số góc, khác hệ số chặn. (Muốn hai nhóm có hệ số góc khác nhau cần thêm số hạng tương tác, chưa học tuần này.)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.8))
luoi = np.linspace(200, 1550, 50)
for ma, nhan, mau, dau in [("hb", "bìa cứng", "tab:blue", "s"), ("pb", "bìa mềm", "tab:olive", "^")]:
    nhom = books[books["cover"] == ma]
    ax.scatter(nhom["volume"], nhom["weight"], marker=dau, color=mau, label=nhan)
    D = 1 if ma == "pb" else 0
    ax.plot(luoi, b0_s + b_vol * luoi + b_D * D, color=mau, linestyle="-" if D == 0 else "--")
ax.set(xlabel="Thể tích (cm³)", ylabel="Khối lượng (g)", title="Hình 4.1. Hai đường song song từ một mô hình")
ax.legend()
plt.show()

> **W04-Q04 · Mức 1 — Mức tham chiếu là gì?** Trong $\widehat{weight} = 197{,}96 + 0{,}72\, volume - 184{,}05\, D$ với $D = 1$ cho bìa mềm, mức tham chiếu của `cover` là gì? Giải thích bằng cách đặt $D = 0$.

<details><summary>Đáp án</summary>

Bìa cứng. Khi $D = 0$, số hạng của `cover` biến mất và mô hình trở thành đường dành cho sách bìa cứng: $\widehat{weight} = 197{,}96 + 0{,}72\, volume$.
</details>

> **W04-WP03 · Mức 2 — Diễn giải ba hệ số.** Với mô hình trên (weight tính bằng g, volume bằng cm³), diễn giải: (1) hệ số `volume`; (2) hệ số `cover`; (3) hệ số chặn.

<details><summary>Lời giải</summary>

1. `volume`: giữ loại bìa cố định, sách lớn hơn 1 cm³ được mô hình dự báo nặng hơn khoảng 0,72 g.
2. `cover`: giữ thể tích cố định, sách bìa mềm được mô hình dự báo nhẹ hơn sách bìa cứng khoảng 184,05 g. Trên Hình 4.1, đó là khoảng cách thẳng đứng giữa hai đường song song, như nhau ở mọi thể tích.
3. Hệ số chặn: giá trị đối chiếu của khối lượng sách bìa cứng có thể tích 0 cm³; không có ý nghĩa thực tế.
</details>

> **W04-EX05 · Mức 2 — Giá trị đối chiếu cho sách bìa mềm.** Một cuốn sách bìa mềm có `volume` = 600 cm³. Tính giá trị đối chiếu của khối lượng.

<details><summary>Đáp án</summary>

Bìa mềm nên $D = 1$: $\widehat{weight} = 197{,}96 + 0{,}72 \times 600 - 184{,}05 = 445{,}91$ g. Đây là giá trị đối chiếu của mô hình, không phải khối lượng chắc chắn của cuốn sách.
</details>

### 4.3 Ví dụ của STAT 20: nhà hàng Ý ở Manhattan

Zagat Guide từng là nguồn đánh giá nhà hàng uy tín ở Mỹ: một nhà phê bình chuyên nghiệp chấm mỗi nhà hàng trên thang 30 điểm cho đồ ăn (`food`), trang trí (`decor`) và dịch vụ (`service`), kèm giá trung bình một bữa (`price`, USD). Dữ liệu `zagat` gồm 168 nhà hàng Ý ở Manhattan, thêm biến `geo` cho biết nhà hàng ở phía đông hay phía tây đảo.

In [ ]:
zagat = pd.read_csv(DATA + "zagat.csv")
print(zagat.shape)
zagat.head()

Câu hỏi đầu tiên: chất lượng đồ ăn liên hệ thế nào với giá, và vị trí có đóng vai trò gì?

In [ ]:
MAU_GEO = {"east": "tab:red", "west": "tab:cyan"}
jit = rng.uniform(-0.15, 0.15, len(zagat))

fig, ax = plt.subplots(figsize=(7, 5))
for vung, nhom in zagat.groupby("geo"):
    ax.scatter(nhom["food"] + jit[nhom.index], nhom["price"], s=18, color=MAU_GEO[vung], label=vung, alpha=0.8)
ax.set(xlabel="Điểm đồ ăn", ylabel="Giá trung bình một bữa (USD)", title="Hình 4.2. Giá theo chất lượng đồ ăn và vị trí")
ax.legend(title="Vị trí")
plt.show()

Muốn ăn ngon thì phải trả tiền: liên hệ tuyến tính dương, khá mạnh. Hai màu trộn lẫn, nhưng nhìn kỹ các nhà hàng phía tây có vẻ rẻ hơn một chút. Mô hình `price ~ food + geo` tóm tắt điều đó bằng hai đường song song.

In [ ]:
zagat["geowest"] = (zagat["geo"] == "west").astype(int)
X = np.column_stack([np.ones(len(zagat)), zagat["food"], zagat["geowest"]])
c0, c_food, c_west = np.linalg.lstsq(X, zagat["price"], rcond=None)[0]
print(f"price^ = {c0:.2f} + {c_food:.3f}·food + ({c_west:.3f})·geowest")

fig, ax = plt.subplots(figsize=(7, 5))
luoi = np.linspace(16, 25.5, 50)
for vung, nhom in zagat.groupby("geo"):
    ax.scatter(nhom["food"] + jit[nhom.index], nhom["price"], s=14, color=MAU_GEO[vung], alpha=0.6, label=vung)
    w = 1 if vung == "west" else 0
    ax.plot(luoi, c0 + c_food * luoi + c_west * w, color=MAU_GEO[vung], linewidth=2)
ax.axvline(18, color="gray", linestyle="--")
ax.annotate(f"khoảng cách = {abs(c_west):.2f} USD", xy=(18, c0 + c_food * 18), xytext=(18.3, 22))
ax.set(xlabel="Điểm đồ ăn", ylabel="Giá (USD)", title="Hình 4.3. price ~ food + geo: hai đường song song")
ax.legend(title="Vị trí")
plt.show()

Mô hình $\widehat{price} = -15{,}97 + 2{,}87\, food - 1{,}46\, geowest$, với mức tham chiếu là `east` (đứng trước `west` theo bảng chữ cái). Diễn giải:

- `food`: với hai nhà hàng **cùng một phía** Manhattan, nhà hàng có điểm đồ ăn cao hơn 1 được dự báo có giá cao hơn khoảng 2,87 USD.
- `geowest`: với hai nhà hàng **cùng điểm đồ ăn**, nhà hàng phía tây được dự báo rẻ hơn nhà hàng phía đông khoảng 1,46 USD. Trên Hình 4.3, đó là khoảng cách thẳng đứng giữa hai đường tại bất kỳ lát cắt nào, ví dụ tại `food` = 18.

Nếu đổi mức tham chiếu sang `west`, mô hình thành $-17{,}43 + 2{,}87\, food + 1{,}46\, geoeast$: cùng hai đường thẳng, chỉ đổi cách viết.

### 4.4 Ba biến số

Mô hình `price ~ food + decor` có một biến phụ thuộc và hai biến giải thích số:

$$
\widehat{price} = -24{,}5 + 1{,}64\, food + 1{,}88\, decor .
$$

Không còn là hai đường thẳng: với hai biến giải thích số, mô hình mô tả một **mặt phẳng** trong không gian ba chiều. Độ nghiêng theo chiều `decor` dốc hơn một chút so với chiều `food`, đúng như hệ số 1,88 lớn hơn 1,64.

In [ ]:
X = np.column_stack([np.ones(len(zagat)), zagat["food"], zagat["decor"]])
d0, d_food, d_decor = np.linalg.lstsq(X, zagat["price"], rcond=None)[0]
print(f"price^ = {d0:.2f} + {d_food:.3f}·food + {d_decor:.3f}·decor")

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(zagat["food"], zagat["decor"], zagat["price"], s=10, color="black")
F, De = np.meshgrid(np.linspace(16, 25, 10), np.linspace(6, 25, 10))
ax.plot_surface(F, De, d0 + d_food * F + d_decor * De, alpha=0.3, color="tab:blue")
ax.set(xlabel="food", ylabel="decor", zlabel="price")
ax.set_title("Hình 4.4. Mặt phẳng hồi quy price ~ food + decor")
ax.view_init(elev=18, azim=-60)
plt.show()

STAT 20 gợi ý hai nhà hàng thú vị để tìm trên hình này: Gennaro có trang trí rất tệ nhưng đồ ăn khá ngon và giá tương xứng; Lamarca trang trí cũng tệ nhưng giá rẻ bất ngờ với đồ ăn khá ổn.

In [ ]:
zagat[zagat["restaurant"].isin(["Gennaro", "Lamarca"])][["restaurant", "price", "food", "decor"]]

> **W04-Q05 · Mức 2 — "Giữ biến còn lại cố định".** Diễn giải hệ số 1,64 của `food` trong $\widehat{price} = -24{,}5 + 1{,}64\, food + 1{,}88\, decor$. Câu trả lời phải có: đối tượng so sánh; 1 đơn vị thay đổi; biến phụ thuộc thay đổi bao nhiêu; điều kiện "giữ biến còn lại cố định".

<details><summary>Đáp án</summary>

So sánh hai nhà hàng có cùng điểm `decor`: nhà hàng có `food` cao hơn 1 điểm được mô hình dự báo với `price` cao hơn khoảng 1,64 USD. Đây là mối liên hệ đã điều chỉnh theo `decor`, không tự động là tác động nhân quả của chất lượng đồ ăn lên giá.
</details>

---
## Phần 5. Diễn giải hệ số và các bẫy thường gặp

### 5.1 Hệ số đơn biến và hệ số đã điều chỉnh không giống nhau

So sánh hai mô hình $Y \sim X_1$ và $Y \sim X_1 + X_2$. Hệ số của $X_1$ có thể thay đổi khi thêm $X_2$, vì **câu hỏi đã đổi**:

- mô hình đơn: mối liên hệ tổng thể giữa $X_1$ và $Y$;
- mô hình nhiều biến: mối liên hệ giữa $X_1$ và $Y$ **ở cùng mức** $X_2$.

Nếu $X_1$ và $X_2$ liên hệ với nhau, mô hình đơn trộn lẫn phần liên hệ riêng của $X_1$ với phần cấu trúc mà $X_1$ chia sẻ cùng $X_2$. Với dữ liệu Zagat, `food` và `decor` có tương quan dương, nên hệ số của `food` đổi rõ khi thêm `decor`:

In [ ]:
for cong_thuc, cot in [("price ~ food", ["food"]), ("price ~ food + decor", ["food", "decor"])]:
    X = np.column_stack([np.ones(len(zagat))] + [zagat[c] for c in cot])
    he_so = np.linalg.lstsq(X, zagat["price"], rcond=None)[0]
    print(f"{cong_thuc:<22} hệ số food = {he_so[1]:.3f}")
print("Tương quan food và decor:", round(zagat["food"].corr(zagat["decor"]), 3))

Trong mô hình đơn, `food` "gánh" cả phần giá mà thực ra đi cùng trang trí đẹp (nhà hàng đồ ăn ngon thường cũng đầu tư trang trí). Khi giữ `decor` cố định, phần đó được tách ra. "Đã điều chỉnh" là một mô tả của mô hình, không phải bảo đảm đã loại bỏ mọi nhiễu.

> **W04-CH02 · Mức 3 — Một đường chung che mất cấu trúc nhóm.** *Dữ liệu minh họa:* Nhóm A: (1, 3), (2, 2), (3, 1); Nhóm B: (4, 13), (5, 12), (6, 11). Bỏ qua nhóm: $\hat Y = -1{,}20 + 2{,}343X$. Giữ nhóm bằng $I(B)$: $\hat Y = 4{,}00 - 1{,}00X + 13{,}00\, I(B)$. Giải thích: (1) vì sao hệ số của $X$ đổi dấu; (2) hệ số −1 đang so sánh những quan sát nào; (3) đường hồi quy chung đã che mất cấu trúc gì.

In [ ]:
nhom_ch = pd.DataFrame({"X": [1, 2, 3, 4, 5, 6], "Y": [3, 2, 1, 13, 12, 11], "B": [0, 0, 0, 1, 1, 1]})
g1, g0 = np.polyfit(nhom_ch["X"], nhom_ch["Y"], 1)
h0, hX, hB = np.linalg.lstsq(np.column_stack([np.ones(6), nhom_ch["X"], nhom_ch["B"]]), nhom_ch["Y"], rcond=None)[0]
print(f"Bỏ qua nhóm: Y^ = {g0:.2f} + {g1:.3f}X")
print(f"Giữ nhóm   : Y^ = {h0:.2f} + ({hX:.2f})X + {hB:.2f}·I(B)")

fig, ax = plt.subplots(figsize=(6, 4.5))
for b, mau, ten in [(0, "tab:blue", "Nhóm A"), (1, "tab:orange", "Nhóm B")]:
    d = nhom_ch[nhom_ch["B"] == b]
    ax.scatter(d["X"], d["Y"], color=mau, s=50, label=ten)
    ax.plot(d["X"], h0 + hX * d["X"] + hB * b, color=mau)
luoi = np.linspace(0.5, 6.5, 20)
ax.plot(luoi, g0 + g1 * luoi, "k--", label="đường chung")
ax.set(xlabel="X", ylabel="Y", title="Hình 5.1. Đảo dấu kiểu Simpson trong hồi quy")
ax.legend()
plt.show()

<details><summary>Lời giải</summary>

1. Khi gộp, nhóm B vừa có $X$ lớn hơn vừa có mức $Y$ nền cao hơn; đường chung chủ yếu nối hai cụm nên có hệ số góc dương 2,343.
2. Trong từng nhóm, mỗi khi $X$ tăng 1 thì $Y$ giảm đúng 1. Hệ số −1 so sánh các quan sát **trong cùng một nhóm**.
3. Đường chung che mất cả hai sự thật: có hai nhóm khác mức nền, và trong mỗi nhóm quan hệ là âm.

Đây là một dạng đảo dấu kiểu Simpson trong hồi quy, nối tiếp W03-CH04 của tuần trước. Không hệ số nào, tự thân, chứng minh tác động nhân quả.
</details>

### 5.2 "Giữ các biến khác cố định" không phải phép màu nhân quả

Một hệ số hồi quy nhiều biến so sánh các quan sát có cùng giá trị của những biến giải thích khác **trong mô hình**. Điều đó không bảo đảm rằng mọi biến gây nhiễu quan trọng đã được đo, rằng dạng tuyến tính là đúng, hay rằng dữ liệu có thiết kế cho phép diễn giải nhân quả. Các điều kiện nhân quả có hệ thống sẽ học ở Tuần 12–13.

Mô tả mối liên hệ trả lời: *các biến cùng thay đổi thế nào trong dữ liệu quan sát?* Câu hỏi nhân quả hỏi: *điều gì sẽ thay đổi nếu ta can thiệp vào một biến, giữ các yếu tố thích hợp khác như cũ?* Tương quan và hồi quy từ dữ liệu quan sát, tự chúng, không đủ để đi từ câu hỏi thứ nhất sang câu hỏi thứ hai.

> **W04-Q06 · Mức 2 — Phản biện một diễn giải sai.** Một sinh viên đọc hệ số `food` = 1,64 trong mô hình Zagat rồi viết: "Tăng điểm chất lượng đồ ăn thêm 1 sẽ làm giá bữa ăn tăng 1,64 đô-la." Sửa câu này cho khớp với thông tin mô hình thật sự cung cấp.

<details><summary>Đáp án</summary>

"So sánh các nhà hàng có cùng điểm `decor`, nhà hàng có điểm `food` cao hơn 1 được mô hình dự báo với giá cao hơn trung bình khoảng 1,64 USD." Câu sửa bỏ ngôn ngữ can thiệp "làm tăng" và thêm điều kiện giữ `decor` cố định.
</details>

> **W04-CH05 · Mức 3 — Cùng đường hồi quy, hai câu chuyện khác nhau.** Một biểu đồ phân tán giữa `bandwidth` (băng thông) và `download_time` (thời gian tải) cho xu hướng âm gần tuyến tính, và hai nghiên cứu khác nhau đều cho gần cùng một đường hồi quy. Hãy xây hai cơ chế sinh dữ liệu hợp lý: (1) một cơ chế mà đường hồi quy chỉ phản ánh mối liên hệ quan sát; (2) một cơ chế mà thay đổi băng thông thực sự làm thay đổi thời gian tải. Chỉ từ biểu đồ phân tán và đường hồi quy, có phân biệt được hai cơ chế không?

Ô dưới **mô phỏng** đúng hai cơ chế đó. Ở cơ chế 1, băng thông **không hề** ảnh hưởng thời gian tải; cả hai đều do "đời máy chủ" quyết định.

In [ ]:
n = 150
# Cơ chế 1: quan sát. Máy mới được cấp băng thông cao VÀ có ổ lưu trữ nhanh.
doi_may = rng.uniform(0, 1, n)
bang_thong_1 = 50 + 150 * doi_may + rng.normal(0, 15, n)
thoi_gian_1 = 12 - 8 * doi_may + rng.normal(0, 0.9, n)          # không có bang_thong trong công thức

# Cơ chế 2: can thiệp. Cùng loại máy, chủ động đặt các mức giới hạn băng thông.
bang_thong_2 = rng.uniform(50, 200, n)
thoi_gian_2 = 12 - 0.053 * (bang_thong_2 - 50) + rng.normal(0, 0.9, n)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, bt, tg, tieu_de in [(axes[0], bang_thong_1, thoi_gian_1, "Cơ chế 1: chỉ là liên hệ"),
                            (axes[1], bang_thong_2, thoi_gian_2, "Cơ chế 2: tác động thật")]:
    b1, b0 = np.polyfit(bt, tg, 1)
    ax.scatter(bt, tg, s=10, alpha=0.6)
    luoi = np.linspace(bt.min(), bt.max(), 30)
    ax.plot(luoi, b0 + b1 * luoi, color="black")
    ax.set(xlabel="Băng thông (Mbps)", title=f"{tieu_de}\nŷ = {b0:.1f} + ({b1:.3f})x")
axes[0].set_ylabel("Thời gian tải (s)")
plt.suptitle("Hình 5.2. Hai cơ chế, hai đường hồi quy gần như trùng nhau (dữ liệu mô phỏng)")
plt.tight_layout()
plt.show()

<details><summary>Lời giải</summary>

1. **Quan sát:** kết nối băng thông cao chủ yếu được cấp cho máy chủ đời mới, và các máy này đồng thời có ổ lưu trữ nhanh hơn. Đường hồi quy âm trộn ảnh hưởng của phần cứng với băng thông. Ở mô phỏng bên trái, băng thông không có trong công thức sinh thời gian tải.
2. **Can thiệp có kiểm soát:** cùng một loại máy chủ được chạy dưới các mức giới hạn băng thông khác nhau do người làm thí nghiệm đặt. Thay đổi băng thông trực tiếp làm thay đổi thời gian tải.

Hai bộ dữ liệu cho biểu đồ phân tán và hệ số hồi quy rất giống nhau, nhưng ý nghĩa nhân quả hoàn toàn khác. Chỉ từ hồi quy mô tả thì không phân biệt được; thông tin về **cách dữ liệu được tạo ra** là thiết yếu (nhớ lại mô phỏng giờ học và GPA ở Tuần 01).
</details>

---
## Phần 6. Thực hành với Python và bài toán tổng hợp

Đề cương dùng R; notebook này dùng Python cho các thao tác tương đương về mặt thống kê. Để ước lượng mô hình theo cú pháp công thức giống `lm()` của R, dùng thư viện `statsmodels` (có sẵn trong Anaconda; nếu thiếu, cài bằng `pip install statsmodels`).

### 6.1 Tính hệ số tương quan

In [ ]:
r = poverty["Poverty"].corr(poverty["Graduates"])
round(r, 3)

Câu hỏi quan trọng không phải cú pháp, mà là: −0,747 tóm tắt điều gì và bỏ sót điều gì?

> **W04-PY01 · Mức 1 — Đọc kết quả Python.** Từ kết quả `r = -0.747`, đánh giá bốn phát biểu: (1) mối liên hệ tuyến tính là âm; (2) đổi `Graduates` từ phần trăm sang tỉ lệ 0–1 sẽ làm $|r|$ nhỏ đi 100 lần; (3) $r$ chứng minh tốt nghiệp trung học làm giảm nghèo; (4) nên xem biểu đồ phân tán trước khi chốt diễn giải.

<details><summary>Đáp án</summary>

1. Đúng.
2. Sai. Tương quan không đổi dưới phép đổi đơn vị tuyến tính với hệ số dương (W04-EX01).
3. Sai. Tương quan không tự thiết lập quan hệ nhân quả.
4. Đúng. Cần kiểm tra dạng phi tuyến, điểm cực đoan và cấu trúc nhóm.
</details>

### 6.2 Hồi quy tuyến tính đơn giản

In [ ]:
import statsmodels.formula.api as smf

m1 = smf.ols("Poverty ~ Graduates", data=poverty).fit()
m1.params

Ta chỉ xem các hệ số cần diễn giải. Bảng `m1.summary()` đầy đủ chứa nhiều mục suy luận (sai số chuẩn, giá trị p, khoảng tin cậy) chưa học; sẽ quay lại ở Tuần 09–11.

Giá trị đối chiếu và phần dư (tương ứng `fitted()` và `resid()` trong R):

In [ ]:
fitted = m1.fittedvalues
resid = m1.resid
kiem_tra = pd.DataFrame({"y": poverty["Poverty"], "ŷ": fitted.round(2), "e": resid.round(2)})
print(kiem_tra.head())
print("y = ŷ + e đúng cho mọi hàng?", np.allclose(poverty["Poverty"], fitted + resid))

> **W04-PY02 · Mức 2 — Phần dư dương nghĩa là gì?** Giả sử `fitted.iloc[i] == 11.36` và `resid.iloc[i] == 5.44`, biến phụ thuộc là `Poverty` tính theo điểm phần trăm. Suy ra giá trị quan sát và diễn giải phần dư.

<details><summary>Đáp án</summary>

$y_i = 11{,}36 + 5{,}44 = 16{,}80$. Phần dư dương 5,44 nghĩa là tỉ lệ nghèo quan sát được cao hơn giá trị đối chiếu của mô hình 5,44 điểm phần trăm.
</details>

### 6.3 Nhiều biến giải thích, biến phân loại

In [ ]:
m2 = smf.ols("price ~ food + decor", data=zagat).fit()
print(m2.params.round(3), "\n")

m3 = smf.ols("weight ~ volume + C(cover)", data=books).fit()
print(m3.params.round(3))

Trong `m2`, hệ số `food` là mối liên hệ với `price` khi giữ `decor` cố định, và ngược lại. Trong `m3`, `C(cover)` yêu cầu phần mềm tự mã hóa các mức thành biến chỉ báo; tên `C(cover)[T.pb]` cho biết biến chỉ báo là của mức `pb`, nên **mức tham chiếu là `hb`** (bìa cứng). Luôn xác định mức tham chiếu trước khi diễn giải hệ số nhóm.

> **W04-PY03 · Mức 2 — Hồi quy đơn và hồi quy nhiều biến.** Bạn chạy `smf.ols("Y ~ X1", data=df)` và `smf.ols("Y ~ X1 + X2", data=df)`, thấy hệ số `X1` khác nhau đáng kể. Phản ứng nào đúng nhất? (a) một trong hai lần tính bị sai; (b) hai hệ số đang trả lời hai câu hỏi điều kiện khác nhau; (c) phải chọn hệ số có trị tuyệt đối lớn hơn; (d) hồi quy nhiều biến tự động cho tác động nhân quả.

<details><summary>Đáp án</summary>

(b). Mô hình đơn và mô hình nhiều biến định nghĩa hai phép so sánh khác nhau (mục 5.1). Không chọn hệ số vì nó lớn hơn, và hồi quy nhiều biến không tự biến mối liên hệ thành tác động nhân quả.
</details>

### 6.4 Bài toán tổng hợp

> **W04-CASE01 · Mức 3.** *Tình huống minh họa.* Với mô hình Zagat $\widehat{price} = -24{,}5 + 1{,}64\, food + 1{,}88\, decor$, xét một nhà hàng có `food` = 24, `decor` = 20, `price` = 60. (1) Tính giá trị đối chiếu và phần dư. (2) Diễn giải hệ số `food`. (3) Nếu chỉ ước lượng `price ~ food`, hệ số `food` có buộc bằng 1,64 không? (4) Có thể kết luận tăng `food` 1 điểm làm giá tăng 1,64 không?

In [ ]:
y_khop = -24.5 + 1.64 * 24 + 1.88 * 20
print("Giá trị đối chiếu:", round(y_khop, 2), "  Phần dư:", round(60 - y_khop, 2))
print("Hệ số food trong price ~ food:", round(smf.ols("price ~ food", data=zagat).fit().params["food"], 3))

<details><summary>Lời giải</summary>

1. $\hat y = -24{,}5 + 1{,}64 \times 24 + 1{,}88 \times 20 = 52{,}46$; $e = 60 - 52{,}46 = 7{,}54$. Giá quan sát cao hơn giá trị đối chiếu từ hai biến giải thích khoảng 7,54 USD.
2. Với cùng `decor`, điểm `food` cao hơn 1 liên hệ với giá trị đối chiếu của giá cao hơn khoảng 1,64 USD.
3. Không. Hệ số từ `price ~ food` (khoảng 2,94 trên dữ liệu này) không giữ `decor` cố định nên trả lời một câu hỏi khác.
4. Không thể chuyển câu (2) thành "nâng `food` sẽ làm `price` tăng" chỉ từ hồi quy trên dữ liệu quan sát.

Đây là khác biệt giữa mối liên hệ tổng thể và mối liên hệ đã điều chỉnh.
</details>

> **W04-CH03 · Mức 3 — Chọn công cụ tóm tắt.** Bạn có ba bộ dữ liệu: A là biểu đồ phân tán gần đường thẳng, không có điểm cực đoan; B có quan hệ rõ dạng chữ U; C phần lớn điểm gần đường thẳng nhưng có một điểm rất xa theo trục $x$. Không tính toán trước. Với mỗi bộ, quyết định: (1) hệ số tương quan có phải tóm tắt đủ tốt không? (2) hồi quy tuyến tính đơn có hợp lý để mô tả không? (3) cần xem thêm điều gì trước khi tin một con số tóm tắt?

Sau khi tự trả lời, chạy ô dưới để xem ba bộ dữ liệu mô phỏng tương ứng.

In [ ]:
xa = np.linspace(0, 10, 25); ya = 2 + 0.8 * xa + rng.normal(0, 0.8, 25)
xb = np.linspace(-3, 3, 25); yb = xb ** 2 + rng.normal(0, 0.6, 25)
xc = np.append(np.linspace(0, 5, 24), 20); yc = np.append(1 + 0.9 * np.linspace(0, 5, 24) + rng.normal(0, 0.4, 24), 0)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, xs, ys, ten in zip(axes, [xa, xb, xc], [ya, yb, yc], ["A", "B", "C"]):
    b1, b0 = np.polyfit(xs, ys, 1)
    luoi = np.linspace(xs.min(), xs.max(), 30)
    ax.scatter(xs, ys, s=15)
    ax.plot(luoi, b0 + b1 * luoi, color="black")
    ax.set_title(f"Bộ {ten}: r = {np.corrcoef(xs, ys)[0, 1]:.2f}")
plt.suptitle("Hình 6.1. Ba bộ dữ liệu, ba mức độ tin cậy của một con số (dữ liệu mô phỏng)")
plt.tight_layout()
plt.show()

<details><summary>Lời giải</summary>

- **A:** tương quan và hồi quy tuyến tính có thể phù hợp, sau khi kiểm tra biểu đồ phần dư.
- **B:** tương quan có thể gần 0 dù quan hệ rất mạnh; đường thẳng bỏ sót cấu trúc phi tuyến.
- **C:** một điểm có đòn bẩy cao có thể đổi mạnh cả tương quan lẫn hệ số góc (ở đây riêng một điểm đó kéo $r$ từ gần 1 xuống gần 0); cần so sánh mô hình có và không có điểm đó.

Quy trình chung: nhìn đồ thị, rồi mới tính tóm tắt hay ước lượng mô hình, kiểm tra phần dư, và cuối cùng diễn giải trong ngữ cảnh.
</details>

---
## Tổng kết

**Những điều mang theo sau Tuần 04**

1. Xem biểu đồ trước khi đọc một con số.
2. $r$ đo hướng và độ mạnh của mối liên hệ **tuyến tính**, không phải mọi dạng quan hệ. $r$ không đổi khi đổi đơn vị với hệ số dương.
3. Bình phương tối thiểu tối thiểu hóa $\sum e_i^2$ và thỏa $\sum e_i = 0$, $\sum x_i e_i = 0$. Nó là phần mở rộng của việc trung bình là hằng số tốt nhất.
4. Trong hồi quy đơn, $b_1 = r\, s_y / s_x$ và đường hồi quy đi qua $(\bar x, \bar y)$.
5. Hệ số góc phải đi kèm đơn vị và ngữ cảnh; hệ số chặn có thể không có ý nghĩa thực tế. Ngoại suy xa phạm vi dữ liệu không đáng tin.
6. Điểm ảnh hưởng mạnh, dạng phi tuyến và cấu trúc nhóm có thể làm một tóm tắt tuyến tính gây hiểu lầm; biểu đồ phần dư giúp phát hiện chúng.
7. Trong hồi quy nhiều biến, mỗi hệ số được đọc khi giữ các biến còn lại cố định; hệ số có thể đổi, thậm chí đổi dấu, khi phép so sánh thay đổi. Biến phân loại vào mô hình qua biến chỉ báo, và phải biết mức tham chiếu.
8. Hệ số hồi quy và tương quan từ dữ liệu quan sát không tự động là tác động nhân quả.

**Chuẩn bị cho Tuần 05.** Tuần 01–04 đi từ dữ liệu, qua mô tả và trực quan hóa, tới các mối liên hệ, và luôn chỉ nói về dữ liệu đang có. Từ Tuần 05, ta cần một ngôn ngữ mới để mô hình hóa sự bất định, trước khi có thể nói điều gì vượt ra ngoài dữ liệu: biến cố, xác suất, xác suất có điều kiện, độc lập, và các quy tắc tính xác suất.

---
*Nguồn: STAT 20 (UC Berkeley), bản dịch tiếng Việt, §2.10–2.12; OpenIntro Statistics 4e, Chương 8–9; Introduction to Modern Statistics 2e, Chương 7–8. Dữ liệu: `allbacks` (gói R `DAAG`); `nyc` / Zagat (gói R `openintro`); `poverty_mo_phong.csv` là dữ liệu mô phỏng khớp thống kê tóm tắt đã công bố của bộ dữ liệu nghèo đói–tốt nghiệp. Dữ liệu ở Hình 3.5, 5.2, 6.1 là mô phỏng; W04-EX02 và W04-CH02 dùng dữ liệu minh họa của bài tập.*